# 04 — Qualidade dos Dados

## 1. Objetivo

Esta etapa tem como objetivo avaliar formalmente a qualidade dos dados disponibilizados para consumo analítico após as etapas de ingestão, tratamento e modelagem do pipeline.

A avaliação será realizada sobre os dados tratados da camada Silver e sobre a estrutura dimensional disponibilizada na camada Gold, verificando se as informações apresentam condições adequadas para responder às perguntas de negócio definidas no projeto.

Serão avaliadas as seguintes dimensões de qualidade:

- **Completude:** identificação de valores nulos ou ausentes nos atributos relevantes;
- **Consistência:** verificação de valores, formatos, categorias e relações compatíveis com as regras definidas para os dados;
- **Unicidade:** identificação de registros duplicados e validação da unicidade das dimensões do modelo;
- **Acurácia:** avaliação da plausibilidade dos valores no contexto do mercado imobiliário e das regras conhecidas do conjunto de dados;
- **Outliers:** identificação de valores potencialmente discrepantes nas principais métricas numéricas.

As verificações realizadas nesta etapa complementam os diagnósticos executados durante a transformação Bronze → Silver. O objetivo não é apenas identificar problemas, mas também demonstrar, por meio de evidências, a qualidade dos dados que serão utilizados nas análises finais.

In [0]:

# CARREGAMENTO DOS DADOS PARA AVALIAÇÃO DE QUALIDADE


# Camada Silver
df_silver = spark.table(
    "mvp_imobiliario.silver.mercado_imobiliario_tratado"
)

# Camada Gold
dim_tempo = spark.table(
    "mvp_imobiliario.gold.dim_tempo"
)

dim_localidade = spark.table(
    "mvp_imobiliario.gold.dim_localidade"
)

dim_transacao = spark.table(
    "mvp_imobiliario.gold.dim_transacao"
)

fato_mercado = spark.table(
    "mvp_imobiliario.gold.fato_mercado_imobiliario"
)

# Quantidade de registros em cada estrutura
print("Quantidade de registros:")
print(f"Silver: {df_silver.count()}")
print(f"dim_tempo: {dim_tempo.count()}")
print(f"dim_localidade: {dim_localidade.count()}")
print(f"dim_transacao: {dim_transacao.count()}")
print(f"fato_mercado_imobiliario: {fato_mercado.count()}")

Quantidade de registros:
Silver: 36933
dim_tempo: 13
dim_localidade: 2129
dim_transacao: 2
fato_mercado_imobiliario: 36933


## 4.1 Avaliação de completude

A completude dos dados será avaliada por meio da identificação de valores nulos nas colunas da camada Silver, utilizada como fonte para a construção do modelo dimensional.

Para cada atributo serão calculadas a quantidade de valores nulos e sua proporção em relação ao total de registros. Essa análise permite identificar campos com ausência de informação e avaliar seu possível impacto nas análises posteriores.

Os valores nulos encontrados nas métricas de preço devem ser interpretados considerando as regras de tratamento aplicadas durante a transformação da camada Bronze para a Silver, na qual valores iguais a zero que representavam ausência de informação foram convertidos para nulos.

In [0]:
from pyspark.sql.functions import col, sum as spark_sum, when

# AVALIAÇÃO DE COMPLETUDE


total_registros = df_silver.count()

resultado_completude = []

for campo in df_silver.columns:
    
    quantidade_nulos = (
        df_silver
        .filter(col(campo).isNull())
        .count()
    )
    
    percentual_nulos = (
        quantidade_nulos / total_registros * 100
    )
    
    resultado_completude.append(
        (
            campo,
            quantidade_nulos,
            round(percentual_nulos, 2)
        )
    )

df_completude = spark.createDataFrame(
    resultado_completude,
    ["campo", "quantidade_nulos", "percentual_nulos"]
)

display(df_completude)

campo,quantidade_nulos,percentual_nulos
mes,0,0.0
bairro,0,0.0
transacao,0,0.0
mediana_m2,3767,10.2
p25_m2,3767,10.2
p75_m2,3767,10.2
anuncios_ativos_media_dia,0,0.0
novos_anuncios,0,0.0
dias_no_mercado_medio,22,0.06
arquivo_origem,0,0.0


### 4.2 Resultado da avaliação de completude

A avaliação de completude demonstrou que a maior parte dos atributos da camada Silver não apresenta valores nulos. Os campos `mes`, `bairro`, `transacao`, `anuncios_ativos_media_dia`, `novos_anuncios`, `arquivo_origem`, `cidade`, `uf` e `data_ingestao` apresentaram 100% de preenchimento nos 36.933 registros analisados.

As métricas `mediana_m2`, `p25_m2` e `p75_m2` apresentaram 3.767 valores nulos cada, correspondentes a aproximadamente 10,20% dos registros. Esse resultado está relacionado tanto à ausência de informação na fonte original quanto ao tratamento realizado na camada Silver. Na camada Bronze haviam sido identificados 3.756 valores nulos e 11 registros iguais a zero em cada uma dessas métricas. Como os valores iguais a zero representavam ausência de informação de preço, eles foram convertidos para nulos durante a transformação, resultando nos 3.767 valores nulos observados na Silver.

O campo `dias_no_mercado_medio` apresentou apenas 22 valores nulos, correspondentes a aproximadamente 0,06% dos registros.

Os valores ausentes foram mantidos como nulos, em vez de serem substituídos artificialmente por médias ou outros valores estimados, preservando a integridade semântica dos dados e evitando a introdução de informações não existentes na fonte original. Nas análises posteriores, esses registros serão considerados de acordo com a disponibilidade das métricas necessárias para cada indicador.

## 4.3 Avaliação de consistência

A consistência será avaliada verificando se os valores armazenados respeitam os domínios e regras esperadas para os atributos do conjunto de dados.

Serão verificadas as relações entre os percentis de preço, a presença de valores negativos nas métricas numéricas, os domínios dos campos categóricos e a coerência das informações geográficas.

In [0]:
from pyspark.sql.functions import col


# CONSISTÊNCIA DAS MÉTRICAS NUMÉRICAS


campos_numericos = [
    "mediana_m2",
    "p25_m2",
    "p75_m2",
    "anuncios_ativos_media_dia",
    "novos_anuncios",
    "dias_no_mercado_medio"
]

print("VALORES NEGATIVOS")
print("-" * 50)

for campo in campos_numericos:
    negativos = df_silver.filter(col(campo) < 0).count()
    print(f"{campo}: {negativos}")

# Verificação da relação esperada entre os percentis:
# p25 <= mediana <= p75

inconsistencias_percentis = (
    df_silver
    .filter(
        col("p25_m2").isNotNull() &
        col("mediana_m2").isNotNull() &
        col("p75_m2").isNotNull() &
        (
            (col("p25_m2") > col("mediana_m2")) |
            (col("mediana_m2") > col("p75_m2"))
        )
    )
)

print("\nCONSISTÊNCIA ENTRE PERCENTIS")
print("-" * 50)
print(
    "Registros em que não ocorre p25_m2 <= mediana_m2 <= p75_m2:",
    inconsistencias_percentis.count()
)

VALORES NEGATIVOS
--------------------------------------------------
mediana_m2: 0
p25_m2: 0
p75_m2: 0
anuncios_ativos_media_dia: 0
novos_anuncios: 0
dias_no_mercado_medio: 0

CONSISTÊNCIA ENTRE PERCENTIS
--------------------------------------------------
Registros em que não ocorre p25_m2 <= mediana_m2 <= p75_m2: 0


### 4.3.1 Resultado da avaliação de consistência numérica

A avaliação das métricas numéricas não identificou valores negativos nos campos `mediana_m2`, `p25_m2`, `p75_m2`, `anuncios_ativos_media_dia`, `novos_anuncios` e `dias_no_mercado_medio`, indicando que os valores respeitam os domínios esperados para essas variáveis.

Também foi verificada a relação lógica entre os indicadores de preço. Considerando somente os registros que possuem valores disponíveis para as três métricas, nenhum registro apresentou violação da condição `p25_m2 ≤ mediana_m2 ≤ p75_m2`.

O resultado indica consistência interna entre os percentis de preço presentes no conjunto de dados, não sendo necessária a aplicação de tratamentos adicionais relacionados a essas regras.

### 4.4 Consistência dos campos categóricos

Além das métricas numéricas, serão avaliados os domínios dos principais atributos categóricos. A verificação permite identificar valores inesperados ou inconsistências de padronização nos campos de cidade, unidade federativa e tipo de transação.

In [0]:

# CONSISTÊNCIA DOS CAMPOS CATEGÓRICOS


print("CIDADES")
df_silver.groupBy("cidade").count().orderBy("cidade").show(truncate=False)

print("UNIDADES FEDERATIVAS")
df_silver.groupBy("uf").count().orderBy("uf").show(truncate=False)

print("TIPOS DE TRANSAÇÃO")
df_silver.groupBy("transacao").count().orderBy("transacao").show(truncate=False)

CIDADES
+--------------+-----+
|cidade        |count|
+--------------+-----+
|curitiba      |1797 |
|recife        |1595 |
|rio-de-janeiro|2975 |
|salvador      |3167 |
|sao-paulo     |27399|
+--------------+-----+

UNIDADES FEDERATIVAS
+---+-----+
|uf |count|
+---+-----+
|ba |3167 |
|pe |1595 |
|pr |1797 |
|rj |2975 |
|sp |27399|
+---+-----+

TIPOS DE TRANSAÇÃO
+---------+-----+
|transacao|count|
+---------+-----+
|aluguel  |13586|
|venda    |23347|
+---------+-----+



### 4.4.1 Resultado da avaliação dos campos categóricos

A avaliação dos atributos categóricos confirmou que os valores presentes na camada Silver estão padronizados e pertencem aos domínios esperados.

O conjunto de dados contempla cinco cidades brasileiras, associadas às respectivas unidades federativas: Salvador (BA), Recife (PE), Curitiba (PR), Rio de Janeiro (RJ) e São Paulo (SP). A maior quantidade de registros está associada a São Paulo, com 27.399 observações.

O atributo `transacao` apresentou somente as duas categorias previstas no conjunto de dados: `aluguel`, com 13.586 registros, e `venda`, com 23.347 registros.

Não foram identificadas categorias inesperadas nos campos avaliados, indicando que a padronização realizada durante a transformação para a camada Silver foi preservada corretamente.

## 4.5 Avaliação de unicidade

A avaliação de unicidade tem como objetivo verificar a existência de registros duplicados de acordo com o grão definido para o modelo dimensional.

O grão adotado representa uma observação mensal do mercado imobiliário para uma determinada combinação de localização e tipo de transação. Dessa forma, será verificada a existência de mais de um registro para a mesma combinação de `mes`, `cidade`, `uf`, `bairro` e `transacao`.

In [0]:
from pyspark.sql.functions import col

# AVALIAÇÃO DE UNICIDADE

chaves_grao = [
    "mes",
    "cidade",
    "uf",
    "bairro",
    "transacao"
]

duplicidades_grao = (
    df_silver
    .groupBy(*chaves_grao)
    .count()
    .filter(col("count") > 1)
)

quantidade_combinacoes_duplicadas = duplicidades_grao.count()

excesso_registros_duplicados = (
    duplicidades_grao
    .selectExpr("sum(count - 1) AS excesso")
    .first()["excesso"]
)

if excesso_registros_duplicados is None:
    excesso_registros_duplicados = 0

print("AVALIAÇÃO DE UNICIDADE")
print("-" * 50)
print(f"Total de registros analisados: {df_silver.count()}")
print(
    f"Combinações do grão com mais de um registro: "
    f"{quantidade_combinacoes_duplicadas}"
)
print(
    f"Registros excedentes associados às duplicidades: "
    f"{excesso_registros_duplicados}"
)

if quantidade_combinacoes_duplicadas > 0:
    print("\nExemplos de combinações duplicadas:")
    display(
        duplicidades_grao
        .orderBy(col("count").desc())
        .limit(20)
    )
else:
    print("\nNenhuma duplicidade encontrada para o grão definido.")

AVALIAÇÃO DE UNICIDADE
--------------------------------------------------
Total de registros analisados: 36933
Combinações do grão com mais de um registro: 0
Registros excedentes associados às duplicidades: 0

Nenhuma duplicidade encontrada para o grão definido.


### 4.5.1 Resultado da avaliação de unicidade

A avaliação de unicidade foi realizada considerando o grão definido para o modelo dimensional: uma observação mensal para cada combinação de cidade, unidade federativa, bairro e tipo de transação.

Foram analisados os 36.933 registros da camada Silver e não foram identificadas combinações duplicadas para os atributos `mes`, `cidade`, `uf`, `bairro` e `transacao`.

O resultado confirma que cada registro representa uma observação única dentro do grão estabelecido para o projeto, não sendo necessária a remoção de registros por duplicidade.

## 4.6 Avaliação de acurácia e plausibilidade

A avaliação de acurácia busca verificar se os valores observados apresentam sentido no contexto do mercado imobiliário analisado.

Como o projeto não utiliza uma segunda fonte independente para validação individual de cada observação, esta etapa não pretende comprovar a exatidão absoluta dos valores. A análise será realizada por meio de verificações de plausibilidade, relações lógicas entre os atributos e investigação dos valores extremos encontrados no conjunto de dados.

In [0]:
from pyspark.sql.functions import col

# AVALIAÇÃO DE ACURÁCIA E PLAUSIBILIDADE

campos_avaliacao = [
    "mediana_m2",
    "p25_m2",
    "p75_m2",
    "anuncios_ativos_media_dia",
    "novos_anuncios",
    "dias_no_mercado_medio"
]

print("RESUMO ESTATÍSTICO DAS MÉTRICAS")
print("-" * 60)

df_silver.select(*campos_avaliacao).summary(
    "count",
    "min",
    "25%",
    "50%",
    "75%",
    "max"
).show(truncate=False)

print("\nREGISTROS COM MAIORES VALORES DE MEDIANA_M2")
display(
    df_silver
    .filter(col("mediana_m2").isNotNull())
    .select(
        "mes",
        "cidade",
        "uf",
        "bairro",
        "transacao",
        "mediana_m2",
        "p25_m2",
        "p75_m2",
        "anuncios_ativos_media_dia",
        "novos_anuncios",
        "dias_no_mercado_medio"
    )
    .orderBy(col("mediana_m2").desc())
    .limit(20)
)

RESUMO ESTATÍSTICO DAS MÉTRICAS
------------------------------------------------------------
+-------+----------+-------+-------+-------------------------+--------------+---------------------+
|summary|mediana_m2|p25_m2 |p75_m2 |anuncios_ativos_media_dia|novos_anuncios|dias_no_mercado_medio|
+-------+----------+-------+-------+-------------------------+--------------+---------------------+
|count  |33166     |33166  |33166  |36933                    |36933         |36911                |
|min    |1         |1      |1      |0                        |0             |0                    |
|25%    |51        |46     |56     |2                        |0             |253                  |
|50%    |3940      |3629   |4214   |5                        |0             |474                  |
|75%    |6291      |5672   |6934   |17                       |1             |776                  |
|max    |3575429   |1191562|7669643|2519                     |327           |4404                 |
+------

mes,cidade,uf,bairro,transacao,mediana_m2,p25_m2,p75_m2,anuncios_ativos_media_dia,novos_anuncios,dias_no_mercado_medio
2026-07-01,sao-paulo,sp,Jardim Itatinga,venda,3575429,7056,7669643,5,0,548
2026-06-01,sao-paulo,sp,Jardim Itatinga,venda,2930087,7404,6283460,6,0,437
2026-05-01,recife,pe,Tejipió,venda,2381041,1191562,3570521,3,0,102
2026-04-01,recife,pe,Tejipió,venda,2381041,1191562,3570521,2,1,93
2026-03-01,recife,pe,Tejipió,venda,2381041,1191562,3570521,2,0,79
2026-05-01,sao-paulo,sp,Jardim Itatinga,venda,2076680,7305,4450979,6,3,413
2026-06-01,recife,pe,Tejipió,venda,1112386,557234,2936193,3,2,115
2025-11-01,sao-paulo,sp,Vila Nova Galvão,venda,1063810,1063810,1063810,1,1,1
2026-09-01,sao-paulo,sp,Não Informado,venda,679874,449329,2854659,90,0,272
2026-08-01,sao-paulo,sp,Não Informado,venda,679874,449329,2854659,90,0,248


### 4.6.1 Investigação dos valores extremos

O resumo estatístico revelou elevada assimetria nas métricas de preço. A mediana geral de `mediana_m2` foi de 3.940, enquanto 75% das observações apresentaram valores de até 6.291. Entretanto, o valor máximo observado para essa variável foi de 3.575.429.

A inspeção dos maiores valores mostrou concentração de observações extremas em determinados bairros e períodos, incluindo Jardim Itatinga, em São Paulo, e Tejipió, em Recife. Alguns desses registros também apresentam pequena quantidade de anúncios ativos, o que pode tornar as estatísticas agregadas mais sensíveis às características específicas dos anúncios disponíveis no período.

Apesar da magnitude elevada, os registros extremos mantêm coerência interna entre as métricas de preço, respeitando a relação `p25_m2 ≤ mediana_m2 ≤ p75_m2`. Dessa forma, somente a análise estatística não fornece evidência suficiente para classificá-los como erros de origem.

Como não existe uma fonte independente no projeto para validar individualmente esses valores, optou-se por preservá-los no conjunto de dados. Os valores serão classificados como potenciais outliers e considerados com cautela durante a análise, evitando sua remoção arbitrária e preservando a rastreabilidade em relação aos dados de origem.

## 4.7 Avaliação de outliers

A identificação de potenciais outliers será realizada pelo método do intervalo interquartil (IQR). Para cada métrica numérica, serão calculados o primeiro quartil (Q1), o terceiro quartil (Q3) e os limites inferior e superior definidos por 1,5 vezes o intervalo interquartil.

A identificação estatística de um outlier não implica automaticamente que o registro seja incorreto. Os resultados serão utilizados como instrumento de diagnóstico e interpretados de acordo com o contexto dos dados.

In [0]:

# AVALIAÇÃO DE OUTLIERS PELO MÉTODO IQR


campos_outliers = [
    "mediana_m2",
    "p25_m2",
    "p75_m2",
    "anuncios_ativos_media_dia",
    "novos_anuncios",
    "dias_no_mercado_medio"
]

resultado_outliers = []

for campo in campos_outliers:

    dados_validos = df_silver.filter(col(campo).isNotNull())

    q1, q3 = dados_validos.approxQuantile(
        campo,
        [0.25, 0.75],
        0.01
    )

    iqr = q3 - q1
    limite_inferior = q1 - (1.5 * iqr)
    limite_superior = q3 + (1.5 * iqr)

    quantidade_outliers = (
        dados_validos
        .filter(
            (col(campo) < limite_inferior) |
            (col(campo) > limite_superior)
        )
        .count()
    )

    total_validos = dados_validos.count()

    percentual = round(
        (quantidade_outliers / total_validos) * 100, 2
    ) if total_validos > 0 else 0

    resultado_outliers.append(
        (
            campo,
            float(q1),
            float(q3),
            float(iqr),
            float(limite_inferior),
            float(limite_superior),
            quantidade_outliers,
            percentual
        )
    )

df_outliers = spark.createDataFrame(
    resultado_outliers,
    [
        "campo",
        "q1",
        "q3",
        "iqr",
        "limite_inferior",
        "limite_superior",
        "quantidade_outliers",
        "percentual_outliers"
    ]
)

display(df_outliers)

campo,q1,q3,iqr,limite_inferior,limite_superior,quantidade_outliers,percentual_outliers
mediana_m2,53.0,6290.0,6237.0,-9302.5,15645.5,803,2.42
p25_m2,47.0,5621.0,5574.0,-8314.0,13982.0,619,1.87
p75_m2,58.0,6832.0,6774.0,-10103.0,16993.0,996,3.0
anuncios_ativos_media_dia,2.0,17.0,15.0,-20.5,39.5,5384,14.58
novos_anuncios,0.0,1.0,1.0,-1.5,2.5,6051,16.38
dias_no_mercado_medio,249.0,764.0,515.0,-523.5,1536.5,2138,5.79


### 4.7.1 Interpretação dos outliers identificados

A aplicação do método do intervalo interquartil (IQR) identificou potenciais outliers em todas as métricas avaliadas. Nas variáveis relacionadas ao preço por metro quadrado, a proporção de valores classificados como extremos foi relativamente baixa: 2,42% em `mediana_m2`, 1,87% em `p25_m2` e 3,00% em `p75_m2`.

As maiores proporções ocorreram nas métricas relacionadas à quantidade de anúncios. Em `anuncios_ativos_media_dia`, 14,58% das observações foram classificadas como potenciais outliers, enquanto em `novos_anuncios` o percentual foi de 16,38%. A variável `dias_no_mercado_medio` apresentou 5,79%.

Esses resultados devem ser interpretados considerando a distribuição e o significado de cada variável. No caso de `novos_anuncios`, por exemplo, o terceiro quartil foi igual a 1 e o limite superior calculado pelo método IQR foi de 2,5. Dessa forma, observações com três ou mais novos anúncios podem ser classificadas estatisticamente como outliers, embora sejam perfeitamente plausíveis no contexto do mercado imobiliário.

Por esse motivo, a identificação pelo método IQR foi utilizada como mecanismo de diagnóstico, e não como critério automático de exclusão. Como os registros analisados não apresentaram valores negativos, mantiveram consistência entre os percentis de preço e não existe no projeto uma fonte independente que permita confirmar que os valores extremos representam erros de origem, optou-se pela preservação dos registros.

Os potenciais outliers serão considerados durante a interpretação das análises posteriores, especialmente nas métricas de preço, nas quais valores extremos podem influenciar medidas agregadas.

## 4.8 Avaliação de acurácia e conclusão da qualidade dos dados

A avaliação de acurácia foi realizada considerando a plausibilidade dos valores no contexto do conjunto de dados e as regras internas disponíveis. Foram verificadas a existência de valores negativos nas métricas numéricas, a coerência entre os percentis de preço (`p25_m2 ≤ mediana_m2 ≤ p75_m2`), a validade das categorias de cidade, unidade federativa e tipo de transação e a presença de valores estatisticamente extremos.

Não foram identificados valores negativos nem inconsistências na relação entre os percentis de preço. Os campos categóricos apresentaram valores compatíveis com o escopo definido para o projeto, composto por cinco cidades brasileiras e pelos tipos de transação venda e aluguel.

Foram identificados valores extremos pelo método IQR, principalmente nas métricas relacionadas à quantidade de anúncios. Entretanto, a classificação estatística como outlier não representa, isoladamente, evidência de erro. Como não existe uma fonte independente para validação individual desses registros, optou-se por preservá-los e documentá-los como potenciais valores extremos.

Em relação à completude, foram identificados valores ausentes nas métricas de preço e em `dias_no_mercado_medio`. Esses casos foram mantidos como nulos quando a informação não estava disponível na fonte, evitando a criação artificial de valores por imputação sem fundamento de negócio.

A avaliação de unicidade não identificou duplicidades considerando o grão definido para o conjunto de dados. Dessa forma, o conjunto foi considerado adequado para prosseguir para a etapa analítica, mantendo documentadas as limitações identificadas.